# Trajectory Preprocessing

Format-agnostic trajectory processing pipeline. Works with any standardized input from `0-data-loaders.ipynb`.

## Processing Steps:
1. **Load standardized data** - Read raw CSV from data loaders
2. **Sampling** - Reduce data density
3. **Outlier cleaning** - Remove GPS errors
4. **Kalman smoothing** - Reduce noise
5. **Stop detection** - Identify stationary periods
6. **Spatial-aware gap splitting** - Split trajectories only when there's BOTH a time gap AND significant location change (prevents splitting during extended indoor visits where GPS was lost)
7. **POI detection** - Query OpenStreetMap for nearby POIs
8. **Export** - Save processed trajectory segments and POI data

In [ ]:
import json
import math
import os
import threading
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import timedelta

import matplotlib.pyplot as plt
import pandas as pd
import requests

try:
    import contextily as ctx
    import geopandas as gpd
    import movingpandas as mpd
    import holoviews as hv
    from holoviews import opts
except ImportError:
    !pip install geopandas movingpandas shapely contextily folium hvplot holoviews
    import contextily as ctx
    import geopandas as gpd
    import movingpandas as mpd
    import holoviews as hv
    from holoviews import opts

warnings.filterwarnings("ignore")

hv.extension("bokeh")

opts.defaults(
    opts.Overlay(active_tools=["wheel_zoom"], frame_width=500, frame_height=400)
)


In [ ]:
# Configuration parameters
CONFIG = {
    'input_dir': '../files/raw',      # Standardized input from data loaders
    # TRAJECTO_CSV_DIR scopes the output per corpus so two ingestions can
    # coexist; notebook 2 reads the same variable.
    'output_dir': os.environ.get('TRAJECTO_CSV_DIR', '../files/csv'),
    'cache_file': '../files/osm_cache.json',

    'stop_detection': {
        'min_duration_seconds': 60,
        'max_diameter_meters': 20
    },
    'osm_query': {
        'radius_meters': 100,
        'max_concurrent_requests': 1,  # Serialize: Overpass rate-limits parallel requests per IP
        'request_delay_seconds': 2.0,  # Delay between requests (concurrency=1 makes this per-request)
    },
    'trajectory': {
        'sampling_rate': 5,  # Take every Nth point
        'gap_threshold_minutes': 30,
        'gap_distance_threshold_km': 0.5,  # Only split if gap AND distance > this threshold
        'min_segment_points': 10,  # Minimum points required for a valid segment
        'outlier_alpha': 2
    },
    'kalman_smoothing': {
        'process_noise_std': 0.1,
        'measurement_noise_std': 10
    },
    'poi_search': {
        'max_closest_pois': 1
    }
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"Output directory: {CONFIG['output_dir']}")
print(f"OSM concurrency: {CONFIG['osm_query']['max_concurrent_requests']} parallel requests")
print(f"Min segment points: {CONFIG['trajectory']['min_segment_points']}")

## 1. Select Data Source

Choose which data source to process (GPX or Geolife).

In [ ]:
# List available data sources
available_sources = []

gpx_raw = os.path.join(CONFIG['input_dir'], 'gpx_raw.csv')
geolife_raw = os.path.join(CONFIG['input_dir'], 'geolife_raw.csv')

if os.path.exists(gpx_raw):
    available_sources.append(('gpx', gpx_raw))
    print("Found: gpx_raw.csv")

if os.path.exists(geolife_raw):
    available_sources.append(('geolife', geolife_raw))
    print("Found: geolife_raw.csv")

if not available_sources:
    print(f"No data sources found in {CONFIG['input_dir']}")
    print("Please run 0-data-loaders.ipynb first.")

In [ ]:
# Select data source to process
# Options: 'gpx', 'geolife', or 'auto' (uses first available)
SELECTED_SOURCE = os.environ.get('TRAJECTO_SOURCE', 'auto')

# Auto-select first available source if set to 'auto'
if SELECTED_SOURCE == 'auto':
    if available_sources:
        SELECTED_SOURCE = available_sources[0][0]
        print(f"Auto-selected source: {SELECTED_SOURCE}")
    else:
        raise ValueError("No data sources available. Run 0-data-loaders.ipynb first.")

# Load selected data
source_file = None
for name, path in available_sources:
    if name == SELECTED_SOURCE:
        source_file = path
        break

if source_file is None:
    raise ValueError(f"Source '{SELECTED_SOURCE}' not found. Available: {[s[0] for s in available_sources]}")

# Load data
raw_df = pd.read_csv(source_file, parse_dates=['timestamp'])
print(f"\nLoaded {SELECTED_SOURCE} data: {len(raw_df)} points")
print(f"Columns: {list(raw_df.columns)}")
print(f"\nUsers: {raw_df['user_id'].nunique()}")
print(f"Trajectories: {raw_df['trajectory_id'].nunique()}")
raw_df.head()

## 2. Utility Functions

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two coordinates using Haversine formula."""
    R = 6371  # Earth's radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

In [ ]:
# OSM query cache with thread-safe operations
_cache_lock = threading.Lock()


def load_cache():
    if os.path.exists(CONFIG['cache_file']):
        with open(CONFIG['cache_file']) as f:
            return json.load(f)
    return {}


def save_cache(cache):
    with _cache_lock, open(CONFIG['cache_file'], 'w') as f:
        json.dump(cache, f, indent=2)


def get_cache_key(lat, lon, radius):
    return f"{lat:.6f},{lon:.6f},{radius}"


osm_cache = load_cache()
print(f"Loaded OSM cache with {len(osm_cache)} entries")

In [ ]:
def get_closest_pois(results, target_lat, target_lon, limit=None):
    """Get the closest POIs from OSM results to a target location."""
    if limit is None:
        limit = CONFIG['poi_search']['max_closest_pois']

    if not results or 'elements' not in results:
        return []

    pois_with_distance = []
    for poi in results['elements']:
        if 'lat' in poi and 'lon' in poi:
            distance_km = calculate_distance(target_lat, target_lon, poi['lat'], poi['lon'])
            distance_m = distance_km * 1000
            enhanced_poi = poi.copy()
            enhanced_poi['distance_km'] = round(distance_km, 3)
            enhanced_poi['distance_m'] = round(distance_m, 1)
            pois_with_distance.append((enhanced_poi, distance_km))

    pois_with_distance.sort(key=lambda x: x[1])
    return [poi for poi, dist in pois_with_distance[:limit]]


# Optional local POI index. When TRAJECTO_BULK_BBOX names a downloaded bbox,
# stop lookups are answered from it instead of the rate-limited Overpass
# endpoint (which 429s heavily on a fresh corpus). Unset -> unchanged behaviour.
_BULK = None
if os.environ.get('TRAJECTO_BULK_BBOX'):
    from helpers.osm_bulk_cache import BulkPOIs, BEIJING, ROME
    _BULK = BulkPOIs({'beijing': BEIJING, 'rome': ROME}[
        os.environ['TRAJECTO_BULK_BBOX'].lower()])
    print(f'Bulk POI index active: {len(_BULK.elements):,} named nodes')


def query_osm_single(lat, lon, radius=None):
    """
    Query OpenStreetMap's Overpass API for a single location.
    Retries on 429/504 with exponential backoff (respecting Retry-After).
    Returns (cache_key, result, from_cache) tuple.
    """
    if radius is None:
        radius = CONFIG['osm_query']['radius_meters']

    cache_key = get_cache_key(lat, lon, radius)

    # Check cache first (thread-safe read)
    with _cache_lock:
        if cache_key in osm_cache:
            return (cache_key, osm_cache[cache_key], True)  # True = from cache

    # Local bulk index, when one is loaded. answer() returns None for points
    # outside the indexed bbox; those fall through to the network below.
    if _BULK is not None:
        hit = _BULK.answer(lat, lon, radius)
        if hit is not None:
            with _cache_lock:
                osm_cache[cache_key] = hit
            return (cache_key, hit, True)

    overpass_url = "http://overpass-api.de/api/interpreter"
    # Overpass answers 406 Not Acceptable to the default python-requests
    # User-Agent. A 406 falls into the "other status: don't retry" branch below,
    # so every uncached stop silently resolves to no POIs instead of failing.
    headers = {'User-Agent': 'Trajecto/1.0 (academic research; semantic trajectory KG)'}
    overpass_query = f"""
    [out:json];
    node(around:{radius},{lat},{lon})->.sn;
    node.sn[name];
    out geom;
    """

    backoff_schedule = [5, 10, 30, 60]  # max 4 retries
    for attempt, base_wait in enumerate([0] + backoff_schedule):
        if base_wait:
            time.sleep(base_wait)

        try:
            response = requests.get(overpass_url, params={'data': overpass_query},
                                    timeout=30, headers=headers)
        except Exception as e:
            if attempt == len(backoff_schedule):
                print(f"  OSM exception for ({lat:.4f}, {lon:.4f}) after {attempt} retries: {e}")
                return (cache_key, None, False)
            print(f"  OSM exception for ({lat:.4f}, {lon:.4f}), retrying: {e}")
            continue

        if response.status_code == 200:
            result = response.json()
            with _cache_lock:
                osm_cache[cache_key] = result
            return (cache_key, result, False)

        if response.status_code in (429, 504):
            if attempt == len(backoff_schedule):
                print(f"  OSM gave up on ({lat:.4f}, {lon:.4f}) after {attempt} retries (last status {response.status_code})")
                return (cache_key, None, False)
            # Honor Retry-After if present (seconds)
            retry_after = response.headers.get('Retry-After')
            if retry_after and retry_after.isdigit():
                wait_override = int(retry_after)
                print(f"  OSM {response.status_code} for ({lat:.4f}, {lon:.4f}); Retry-After={wait_override}s")
                time.sleep(max(0, wait_override - base_wait))
            else:
                print(f"  OSM {response.status_code} for ({lat:.4f}, {lon:.4f}); backoff retry {attempt+1}/{len(backoff_schedule)}")
            continue

        # Other status: don't retry
        print(f"  OSM error {response.status_code} for ({lat:.4f}, {lon:.4f})")
        return (cache_key, None, False)

    return (cache_key, None, False)


def query_osm(lat, lon, radius=None):
    """Single location query (backwards compatible)."""
    cache_key, result, from_cache = query_osm_single(lat, lon, radius)
    if from_cache:
        print(f"Cache hit for ({lat:.4f}, {lon:.4f})")
    else:
        print(f"Queried OSM for ({lat:.4f}, {lon:.4f})")
    return result


def query_osm_batch(locations, radius=None):
    """
    Query OSM for multiple locations concurrently.

    Args:
        locations: List of (lat, lon, stop_type, group_info) tuples
        radius: Search radius in meters

    Returns:
        Dict mapping (lat, lon) to OSM results
    """
    if radius is None:
        radius = CONFIG['osm_query']['radius_meters']

    results = {}
    cache_hits = 0
    api_calls = 0

    # Separate cached vs uncached locations
    uncached = []
    for loc in locations:
        lat, lon = loc[0], loc[1]
        cache_key = get_cache_key(lat, lon, radius)
        with _cache_lock:
            if cache_key in osm_cache:
                results[(lat, lon)] = osm_cache[cache_key]
                cache_hits += 1
            else:
                uncached.append(loc)

    if cache_hits > 0:
        print(f"  Cache hits: {cache_hits}/{len(locations)}")

    if not uncached:
        return results

    # Query uncached locations concurrently
    max_workers = CONFIG['osm_query']['max_concurrent_requests']
    print(f"  Querying {len(uncached)} locations ({max_workers} concurrent)...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(query_osm_single, loc[0], loc[1], radius): loc
            for loc in uncached
        }

        for future in as_completed(futures):
            loc = futures[future]
            lat, lon = loc[0], loc[1]
            try:
                cache_key, result, from_cache = future.result()
                if result is not None:
                    results[(lat, lon)] = result
                    api_calls += 1
            except Exception as e:
                print(f"  Error for ({lat:.4f}, {lon:.4f}): {e}")

    # Save cache after batch
    if api_calls > 0:
        save_cache(osm_cache)
        print(f"  API calls: {api_calls}, cache updated")

    # Small delay after batch to be nice to the API
    if api_calls > 0:
        time.sleep(CONFIG['osm_query']['request_delay_seconds'])

    return results

In [ ]:
def filter_relevant_tags(pois):
    """Filter POI data to keep only relevant tags."""
    if not pois:
        return pois

    filtered_pois = []

    for poi in pois:
        if poi is None or not poi:
            continue

        poi_data = poi.get('pois', poi)
        if poi_data is None:
            continue

        if 'elements' in poi_data:
            for element in poi_data['elements']:
                filtered_entry = {
                    'type': element.get('type'),
                    'id': element.get('id'),
                    'lat': element.get('lat'),
                    'lon': element.get('lon'),
                }

                tags = element.get('tags', {})
                relevant_tags = {
                    'amenity': tags.get('amenity'),
                    'brand_en': tags.get('brand:en'),
                    'name_en': tags.get('name:en'),
                    'operator_en': tags.get('operator:en'),
                    'brand_wikidata': tags.get('brand:wikidata'),
                    'operator_wikidata': tags.get('operator:wikidata'),
                    'name': tags.get('name'),
                    'tourism': tags.get('tourism'),
                    'shop': tags.get('shop'),
                    'information': tags.get('information'),
                    'natural': tags.get('natural'),
                    'wikidata': tags.get('wikidata'),
                    'wikipedia': tags.get('wikipedia'),
                    'all_tags': tags
                }
                relevant_tags = {k: v for k, v in relevant_tags.items() if v is not None}
                filtered_entry.update(relevant_tags)
                filtered_pois.append(filtered_entry)

    return filtered_pois

## 3. Trajectory Processing Functions

In [ ]:
def create_geodataframe(df):
    """Convert DataFrame to GeoDataFrame."""
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs="EPSG:4326"
    )
    return gdf

In [ ]:
def mark_stops(trajectory):
    """Mark the detected stops in the trajectory data."""
    detector = mpd.TrajectoryStopDetector(trajectory)
    stops = detector.get_stop_points(
        min_duration=timedelta(seconds=CONFIG['stop_detection']['min_duration_seconds']),
        max_diameter=CONFIG['stop_detection']['max_diameter_meters']
    )

    trajectory.df['is_stop'] = False
    trajectory.df['stop_id'] = ""

    counter = 1
    stop_id_mapping = {}

    trajectory.df['is_stop'] = trajectory.df.geometry.index.isin(stops.index)

    for index, _row in trajectory.df.iterrows():
        for stop_index, stop_row in stops.iterrows():
            if stop_row.start_time <= index <= stop_row.end_time:
                trajectory.df.loc[index, 'is_stop'] = True
                if stop_index not in stop_id_mapping:
                    stop_id_mapping[stop_index] = counter
                    counter += 1
                trajectory.df.loc[index, 'stop_id'] = stop_id_mapping[stop_index]
                break

    return trajectory

In [ ]:
def split_on_significant_gaps(trajectory, gap_minutes=None, distance_threshold_km=None, min_points=None):
    """
    Split trajectory only when there's BOTH a temporal gap AND significant distance change.

    This prevents splitting when someone is stationary (e.g., at a restaurant, indoors)
    but GPS signal was lost. If the user didn't move significantly, we keep it as one trajectory.

    Args:
        trajectory: MovingPandas Trajectory object
        gap_minutes: Minimum time gap to consider (default from config)
        distance_threshold_km: Minimum distance to trigger split (default from config)
        min_points: Minimum points required for a valid segment (default from config)

    Returns:
        List of Trajectory objects (segments) with at least min_points each
    """
    if gap_minutes is None:
        gap_minutes = CONFIG['trajectory']['gap_threshold_minutes']
    if distance_threshold_km is None:
        distance_threshold_km = CONFIG['trajectory']['gap_distance_threshold_km']
    if min_points is None:
        min_points = CONFIG['trajectory']['min_segment_points']

    df = trajectory.df.copy()

    if len(df) < min_points:
        print(f"    Skipped: only {len(df)} points (min: {min_points})")
        return []

    # Calculate time differences between consecutive points
    timestamps = df.index.to_series()
    time_diffs = timestamps.diff()

    # Find potential split points (time gap exceeds threshold)
    gap_threshold = timedelta(minutes=gap_minutes)
    potential_splits = time_diffs[time_diffs > gap_threshold]

    if len(potential_splits) == 0:
        return [trajectory]

    # For each potential split, check if there's also significant distance change
    actual_split_indices = []

    for split_idx in potential_splits.index:
        # Get the index position
        idx_pos = df.index.get_loc(split_idx)

        if idx_pos == 0:
            continue

        # Get coordinates before and after the gap
        prev_idx = df.index[idx_pos - 1]

        lat1 = df.loc[prev_idx, 'latitude']
        lon1 = df.loc[prev_idx, 'longitude']
        lat2 = df.loc[split_idx, 'latitude']
        lon2 = df.loc[split_idx, 'longitude']

        # Calculate distance
        distance_km = calculate_distance(lat1, lon1, lat2, lon2)

        # Only split if distance exceeds threshold
        if distance_km > distance_threshold_km:
            actual_split_indices.append(split_idx)
            gap_hours = time_diffs[split_idx].total_seconds() / 3600
            print(f"    Split point: {gap_hours:.1f}h gap, {distance_km:.2f}km distance")
        else:
            gap_hours = time_diffs[split_idx].total_seconds() / 3600
            print(f"    Kept together: {gap_hours:.1f}h gap but only {distance_km*1000:.0f}m distance")

    if len(actual_split_indices) == 0:
        return [trajectory]

    # Create trajectory segments
    segments = []
    segment_start = df.index[0]
    skipped_segments = 0

    for split_idx in actual_split_indices:
        # Get the index position
        idx_pos = df.index.get_loc(split_idx)
        segment_end = df.index[idx_pos - 1]

        # Extract segment data
        segment_df = df.loc[segment_start:segment_end].copy()

        # Only keep segments with enough points
        if len(segment_df) >= min_points:
            segment_traj = mpd.Trajectory(
                segment_df,
                traj_id=f"{trajectory.id}_{len(segments)}"
            )
            segments.append(segment_traj)
        else:
            skipped_segments += 1

        segment_start = split_idx

    # Add final segment (only if it has enough points)
    final_df = df.loc[segment_start:].copy()
    if len(final_df) >= min_points:
        final_traj = mpd.Trajectory(
            final_df,
            traj_id=f"{trajectory.id}_{len(segments)}"
        )
        segments.append(final_traj)
    else:
        skipped_segments += 1

    if skipped_segments > 0:
        print(f"    Skipped {skipped_segments} segment(s) with < {min_points} points")

    return segments if segments else []

In [ ]:
def find_pois_near_stops(df):
    """
    Find POIs near stop points, grouping consecutive stops.
    Also treats the first and last points as potential origin/destination POIs.
    Uses batch concurrent queries for better performance.
    """
    pois_near_stops = []
    df['poi_ids'] = [[] for _ in range(len(df))]
    df['time_spent'] = [0.0 for _ in range(len(df))]

    # Get first and last indices
    all_indices = list(df.index)
    if len(all_indices) < 2:
        return pois_near_stops

    first_idx = all_indices[0]
    last_idx = all_indices[-1]

    # Find consecutive stops
    stop_groups = []
    current_group = []

    for index, row in df.iterrows():
        if row['is_stop']:
            current_group.append(index)
        elif current_group:
            stop_groups.append(current_group)
            current_group = []

    if current_group:
        stop_groups.append(current_group)

    # Add first point as origin if not already a stop
    first_in_stop = any(first_idx in group for group in stop_groups)
    if not first_in_stop:
        stop_groups.insert(0, [first_idx])
        df.at[first_idx, 'is_stop'] = True
        df.at[first_idx, 'stop_id'] = 'origin'
        print("  Added first point as origin stop")

    # Add last point as destination if not already a stop
    last_in_stop = any(last_idx in group for group in stop_groups)
    if not last_in_stop:
        stop_groups.append([last_idx])
        df.at[last_idx, 'is_stop'] = True
        df.at[last_idx, 'stop_id'] = 'destination'
        print("  Added last point as destination stop")

    # Prepare batch query locations
    query_locations = []
    group_metadata = []  # Store metadata for each group

    for group in stop_groups:
        if not group:
            continue

        group_rows = df.loc[group]
        avg_lat = group_rows['latitude'].mean()
        avg_lon = group_rows['longitude'].mean()

        # Calculate time spent - group contains timestamp index values directly
        if len(group) > 1:
            start_time = group[0]   # Already a timestamp from the index
            end_time = group[-1]
            time_spent = (end_time - start_time).total_seconds()
        else:
            time_spent = 0.0

        # Determine stop type
        if group[0] == first_idx and len(group) == 1 and not first_in_stop:
            stop_type = "origin"
        elif group[-1] == last_idx and len(group) == 1 and not last_in_stop:
            stop_type = "destination"
        else:
            stop_type = "stop"

        query_locations.append((avg_lat, avg_lon, stop_type, group))
        group_metadata.append({
            'group': group,
            'avg_lat': avg_lat,
            'avg_lon': avg_lon,
            'time_spent': time_spent,
            'stop_type': stop_type
        })

    if not query_locations:
        return pois_near_stops

    print(f"  Querying POIs for {len(query_locations)} stop(s)...")

    # Batch query all locations concurrently
    osm_results = query_osm_batch(query_locations)

    # Process results
    for meta in group_metadata:
        group = meta['group']
        avg_lat = meta['avg_lat']
        avg_lon = meta['avg_lon']
        time_spent = meta['time_spent']
        stop_type = meta['stop_type']

        poi_data = osm_results.get((avg_lat, avg_lon))

        if poi_data:
            closest_poi_data = get_closest_pois(poi_data, avg_lat, avg_lon)

            if closest_poi_data:
                poi_ids = [f"{element['id']}-{element['distance_m']}" for element in closest_poi_data]

                for idx in group:
                    df.at[idx, 'poi_ids'] = poi_ids
                    df.at[idx, 'time_spent'] = time_spent

                pois_near_stops.append({
                    'indices': group,
                    'avg_lat': avg_lat,
                    'avg_lon': avg_lon,
                    'time_spent': time_spent,
                    'stop_type': stop_type,
                    'pois': poi_data
                })

    return pois_near_stops

In [ ]:
def plot_trajectory(trajectory):
    """Plot trajectory with OpenStreetMap basemap."""
    fig, ax = plt.subplots(figsize=(10, 10))
    trajectory.plot(ax=ax, linewidth=2, color='blue')
    ctx.add_basemap(ax, crs=trajectory.df.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.show()

## 4. Main Processing Pipeline

In [ ]:
# Group data by trajectory_id
trajectory_groups = raw_df.groupby('trajectory_id')
min_points = CONFIG['trajectory']['min_segment_points']

print(f"Processing {len(trajectory_groups)} trajectories...")
print(f"Minimum points required: {min_points}")
print(f"\n{'='*70}")

# Store trajectory info
trajectories_to_process = []

for traj_id, traj_df in trajectory_groups:
    traj_df = traj_df.sort_values('timestamp').reset_index(drop=True)
    user_id = traj_df['user_id'].iloc[0]

    if len(traj_df) < min_points:
        print(f"Skipping {traj_id}: too few points ({len(traj_df)})")
        continue

    trajectories_to_process.append({
        'trajectory_id': traj_id,
        'user_id': user_id,
        'data': traj_df
    })

print(f"\nTrajectories to process: {len(trajectories_to_process)}")

In [ ]:
# Process each trajectory
print(f"\n{'='*70}")
print("TRAJECTORY PROCESSING")
print(f"{'='*70}\n")

processed_trajectories = []
min_points = CONFIG['trajectory']['min_segment_points']

for traj_info in trajectories_to_process:
    traj_id = traj_info['trajectory_id']
    user_id = traj_info['user_id']
    traj_df = traj_info['data']

    print(f"\nProcessing: {traj_id} (user: {user_id})")

    try:
        # Sample points
        sampled_df = traj_df.iloc[::CONFIG['trajectory']['sampling_rate']].copy()

        if len(sampled_df) < min_points:
            print(f"  Skipping: too few points after sampling ({len(sampled_df)} < {min_points})")
            continue

        print(f"  Points: {len(traj_df)} -> {len(sampled_df)} (after sampling)")

        # Create GeoDataFrame
        gdf = create_geodataframe(sampled_df)
        gdf = gdf.set_index('timestamp')

        # Create trajectory
        my_traj = mpd.Trajectory(gdf, traj_id=traj_id)

        # Clean outliers (with error handling for edge cases)
        print("  Cleaning outliers...")
        try:
            cleaned = mpd.OutlierCleaner(my_traj).clean(alpha=CONFIG['trajectory']['outlier_alpha'])
        except ValueError as e:
            if "at least two rows" in str(e):
                print("  Skipping: outlier cleaning failed (insufficient data)")
                continue
            raise

        # Check if enough points remain after outlier cleaning
        if len(cleaned.df) < min_points:
            print(f"  Skipping: too few points after outlier cleaning ({len(cleaned.df)} < {min_points})")
            continue

        # Apply Kalman smoothing
        print("  Applying Kalman smoothing...")
        try:
            smooth = mpd.KalmanSmootherCV(cleaned).smooth(
                process_noise_std=CONFIG['kalman_smoothing']['process_noise_std'],
                measurement_noise_std=CONFIG['kalman_smoothing']['measurement_noise_std']
            )
        except ValueError as e:
            if "at least two rows" in str(e):
                print("  Skipping: Kalman smoothing failed (insufficient data)")
                continue
            raise

        smooth.add_speed(overwrite=True, units=("km", "h"))

        # Mark stops
        print("  Detecting stops...")
        trajectory_with_stops = mark_stops(smooth)

        # Split on significant gaps (time gap + distance change)
        # This prevents splitting when user is stationary but GPS signal was lost
        print("  Splitting on significant gaps (time + distance)...")
        segments = split_on_significant_gaps(trajectory_with_stops)

        if not segments:
            print("  Skipping: no valid segments after splitting")
            continue

        num_segments = len(segments)
        print(f"  Created {num_segments} segment(s)")

        processed_trajectories.append({
            'trajectory_id': traj_id,
            'user_id': user_id,
            'segments': segments
        })

    except Exception as e:
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*70}")
print(f"PROCESSED {len(processed_trajectories)} TRAJECTORIES")
total_segments = sum(len(pt['segments']) for pt in processed_trajectories)
print(f"TOTAL SEGMENTS: {total_segments}")
print(f"{'='*70}")

In [ ]:
# Find POIs and export
print(f"\n{'='*70}")
print("POI DETECTION AND EXPORT")
print(f"{'='*70}\n")

unique_pois = {}  # Use dict to track unique POIs by ID
saved_files = []
segment_counter = 0
poi_filename = os.path.join(CONFIG['output_dir'], 'pois.csv')


def save_pois_incremental(pois_dict, filepath):
    """Save POIs to CSV incrementally."""
    if pois_dict:
        poi_df = pd.DataFrame(list(pois_dict.values()))
        poi_df.to_csv(filepath, index=False)
        return len(poi_df)
    return 0


for proc_traj in processed_trajectories:
    traj_id = proc_traj['trajectory_id']
    user_id = proc_traj['user_id']

    print(f"\nProcessing POIs for: {traj_id}")

    for segment_idx, traj_segment in enumerate(proc_traj['segments']):
        try:
            # Find POIs near stops
            print(f"  Segment {segment_idx + 1}/{len(proc_traj['segments'])}")
            pois = find_pois_near_stops(traj_segment.df)
            updated_pois = filter_relevant_tags(pois)

            # Prepare dataframe for export - reset index to get timestamp
            export_df = traj_segment.df.reset_index()
            if 'index' in export_df.columns:
                export_df = export_df.rename(columns={'index': 'timestamp'})

            # Split timestamp into date and time columns
            if 'timestamp' in export_df.columns:
                export_df['date'] = export_df['timestamp'].dt.strftime('%Y-%m-%d')
                export_df['time'] = export_df['timestamp'].dt.strftime('%H:%M:%S')
                export_df = export_df.drop(columns=['timestamp'])

            # Add metadata
            export_df['id'] = range(1, len(export_df) + 1)
            export_df['traj_id'] = f"{int(time.time())}{segment_counter}"
            export_df['username'] = user_id
            export_df['source_trajectory'] = traj_id
            export_df['segment_index'] = segment_idx

            # Save trajectory segment to CSV
            safe_traj_id = str(traj_id).replace('/', '_').replace('\\', '_')
            filename = f"{SELECTED_SOURCE}_{safe_traj_id}_{segment_idx + 1}.csv"
            filepath = os.path.join(CONFIG['output_dir'], filename)
            export_df.to_csv(filepath, index=False)
            saved_files.append(filepath)
            print(f"  Saved: {filename}")

            # Collect POIs and save incrementally
            new_pois = 0
            for poi in updated_pois:
                if 'id' in poi and poi['id'] not in unique_pois:
                    poi['source_trajectory'] = traj_id
                    unique_pois[poi['id']] = poi
                    new_pois += 1

            # Save POIs after each segment (incremental update)
            if new_pois > 0:
                total_pois = save_pois_incremental(unique_pois, poi_filename)
                print(f"  POIs: +{new_pois} new (total: {total_pois})")

            segment_counter += 1

        except Exception as e:
            print(f"  Error in segment {segment_idx + 1}: {e}")
            continue

# Final POI save (ensure everything is saved)
if unique_pois:
    total_pois = save_pois_incremental(unique_pois, poi_filename)
    print(f"\nFinal POI count: {total_pois} unique POIs saved to pois.csv")

In [ ]:
# Final summary
print(f"\n{'='*70}")
print("PREPROCESSING COMPLETE")
print(f"{'='*70}")
print(f"\nSource: {SELECTED_SOURCE}")
print(f"Trajectory files saved: {len(saved_files)}")
print(f"POIs collected: {len(unique_pois) if unique_pois else 0}")
print(f"Output directory: {CONFIG['output_dir']}")
print("\nNext step: Run 2-update_rules_and_generate_rdf.ipynb")